# Hoe goed beschrijft de Robertson Soil Behavior Type o.b.v. CPT's onze Vlaamse veenmonsters?

Een praktijkvoorbeeld om via pydov de DOV databank te bevragen om zo aan de hand van monsters, observaties, boringen en sonderingen de Robertson Soil Behavior Type te toetsen aan geobserveerde data.

We halen eerst alle observaties op waarbij er een hoog gehalte (minstens 50%) organische stof gemeten is. Hierbij halen we ook de diepte onder het maaiveld op. Vervolgens halen we via het bijhorende monster ook de gegevens van de boring op, met name het aanvangspeil in mTAW. Zo kunnen we de diepte van het monster in absolute waarde (mTAW) bepalen.

Daarna voeren we een geografische zoekopdracht uit om sonderingen te vinden die zich in de buurt (binnen de 5 meter) van deze veenmonsters bevinden. Zo kunnen we voor elk van de monsters efficiënt de meest nabije sondering opvragen. Ook de sondeerdata herrekenen we naar absolute hoogtewaarden in meter TAW, zodat we enkel het stuk dat planimetrisch en altimetrisch overeenstemt met het veenmonster kunnen weerhouden voor verdere berekeningen.

Vervolgens wordt voor elk monster (observatie) de Robertson Soil Behavior Type analyse uitgevoerd op basis van de beschikbare sondeerdata. De resultaten worden tenslotte op grafiek weergegeven.

### Imports

Eerst importeren we de nodige klassen en bibliotheken:

In [ ]:
from pydov.search.observatie import ObservatieSearch
from pydov.search.monster import MonsterSearch
from pydov.search.sondering import SonderingSearch
from pydov.search.boring import BoringSearch

from pydov.types.observatie import Observatie
from pydov.types.sondering import Sondering

from pydov.search.fields import GeometryReturnField

from pydov.util.location import WithinDistance, GeopandasFilter
from pydov.util.query import Join

from owslib.fes2 import PropertyIsEqualTo

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

### Zoekobjecten

Vervolgens maken we voor elke dataset die we nodig hebben een zoekobject. Met deze zoekobjecten kunnen we vervolgens elk van de datasets bevragen.

In [ ]:
observatie_search = ObservatieSearch()
monster_search = MonsterSearch()
sondering_search = SonderingSearch()
boring_search = BoringSearch()

We hebben niet van alle datasets alle velden nodig: hier definiëren we per dataset welke velden we willen opvragen.

Van de observaties en de sonderingen hebben we naast de standaardvelden ook de geometrie nodig, om later de spatial join te kunnen uitvoeren.

De sonderingen zelf halen we in twee fasen op: eerst enkel de permanente key en de geometrie (nodig voor de spatial join), en pas later de volledige sondeerdata.

### Data opvragen

#### Observaties van veen

Voor de observaties hebben we naast de standaardvelden ook de geometrie nodig, die voegen we toe:

In [ ]:
observatie_fields = Observatie.get_field_names()
observatie_fields.extend([GeometryReturnField(f.name, 31370) for f in observatie_search.get_fields(type='geometry').values()])

observatie_fields

Nu kunnen we starten met het ophalen van de nodige gegevens. We halen eerst de observaties van de parameter 'Gehalte Organische stoffen' op. Het resultaat hiervan is een percentage en zetten we dus om naar getallen (`float`), om zo enkel die met een percentage van minstens 50 procent te weerhouden.

We voegen ook een kolom toe met de `pkey_monster` waaraan de observaties gekoppeld zijn. Wanneer er observaties aan andere objecten dan monsters gekoppeld zouden zijn blijft deze kolom leeg.

In [ ]:
df_observaties = observatie_search.search(
    query= PropertyIsEqualTo('parameter', 'Gehalte Organische stoffen (Gehalte Organische stoffen)'),
    return_fields=observatie_fields
)
df_observaties['resultaat'] = df_observaties['resultaat'].astype(float)
df_observaties = df_observaties[df_observaties['resultaat'] >= 50]
df_observaties['pkey_monster'] =  df_observaties['pkey_parent'].apply(lambda x: x if 'monster' in x else np.nan)

df_observaties

Van deze observaties hebben we hier reeds de (relatieve) diepte beschikbaar, maar om dit om te kunnen rekenen naar absolute hoogten hebben we ook het aanvangspeil (maaiveld) nodig. Dit kunnen we vinden via het monster en de boring.

We halen eerst de monsters op waaraan deze observaties gekoppeld zijn en voegen een kolom toe met de `pkey_boring`. Bij monsters die aan meerdere boringen gekoppeld zouden zijn wordt de eerste weerhouden, bij monsters die aan andere objecten gekoppeld zouden zijn blijft deze kolom leeg.

In [ ]:
df_monsters = monster_search.search(
    query=Join(df_observaties, on='pkey_monster'),
    return_fields=['pkey_monster', 'pkey_parents']
)
df_monsters['pkey_boring'] =  df_monsters['pkey_parents'].apply(lambda x: [i for i in x if 'boring' in i][0])
df_monsters

Op basis van deze link naar de boringen kunnen we tenslotte het aanvangspeil van de boring opvragen:

In [ ]:
df_boringen = boring_search.search(
    query=Join(df_monsters, on='pkey_boring'),
    return_fields=['pkey_boring', 'start_boring_mtaw']
)
df_boringen

Vervolgens kunnen we alle resultaten terug samenvoegen in één dataframe met `pd.merge()`. We voegen eerste de monsters en de boringen samen op basis van de `pkey_boring`, en vervolgens dit resultaat met de observaties op basis van de `pkey_monster`:

In [ ]:
df_monsters = pd.merge(df_monsters, df_boringen, 'outer', 'pkey_boring')
df_observaties = pd.merge(df_observaties, df_monsters, 'outer', 'pkey_monster')

df_observaties

Hiervan maken we dan een GeoPandas GeoDataFrame, om te kunnen gebruiken in een pydov zoekopdracht en om later te kunnen gebruiken in de spatial join met de sonderingen.

In [ ]:
geo_df_observaties = gpd.GeoDataFrame(df_observaties, geometry='geom', crs='EPSG:31370')
geo_df_observaties.explore()

#### Sonderingen in de buurt

In eerste instantie willen we enkel een dataframe opbouwen van alle nabijgelegen sonderingen en hun locatie, maar nog zonder de volledige sondeergegevens. We vragen dus nu eerst enkel de permanente url en de locatie op:

In [ ]:
sondering_fields_base = [
    'pkey_sondering',
]

sondering_fields_geom = [GeometryReturnField(f.name, 31370) for f in observatie_search.get_fields(type='geometry').values()]

sondering_fields_base.extend(sondering_fields_geom)

sondering_fields_base

Nu we alle veenmonsters ter beschikking hebben kunnen we op zoek naar alle nabijgelegen sonderingen. Dit kan heel eenvoudig door gebruik te maken van de `GeopandasFilter` in pydov: via volgende opdracht kunnen we alle sonderingen vinden die zich op 5 meter of minder bevinden van één van de eerder gevonden veenmonsters:

In [ ]:
df_sonderingen_base = sondering_search.search(
    location=GeopandasFilter(geo_df_observaties, WithinDistance, {'distance': 5}),
    query=PropertyIsEqualTo('sondeermethode', 'continu elektrisch'),
    return_fields=sondering_fields_base
)
df_sonderingen_base

Ook hiervan maken we een GeoDataFrame:

In [ ]:
geo_df_sonderingen_base = gpd.GeoDataFrame(df_sonderingen_base, geometry='geom', crs='EPSG:31370')
geo_df_sonderingen_base.explore()

Nu kunnen we beide GeoDataFrames koppelen op basis van hun locatie: zo vinden we voor elke observatie welke sondering(en) zich in de buurt bevinden, en ook wat hun onderlinge afstand is:

In [ ]:
df_joined = gpd.sjoin_nearest(geo_df_observaties, geo_df_sonderingen_base, max_distance=5, distance_col='dist')
df_joined

Nu hebben we enkel nog de sondeergegevens zelf nodig: deze kunnen we opvragen op basis van de eerder gevonden sonderingen, maar nu met de volledige lijst van velden als `return_field`:

In [ ]:
sondering_fields = Sondering.get_field_names()
sondering_fields.extend(sondering_fields_geom)

sondering_fields

In [ ]:
df_sonderingen = sondering_search.search(
    query=Join(df_sonderingen_base, 'pkey_sondering'),
    return_fields=sondering_fields
)
df_sonderingen

### Data analyse

Nu we alle gegevens hebben kunnen we onze analyse aanvatten. We maken eerst een leeg dataframe aan met de nodige kolommen om later de resultaten in te kunnen opslaan:

In [ ]:
new_columns = [
    'pkey_observatie',
    "qc_avg",
    "qc_med",
    "qc_var",
    "fs_avg",
    "fs_med",
    "fs_var",
    "rf_avg",
    "rf_med",
    "rf_var",
    "Qtn_avg",
    "Qtn_med",
    "Qtn_var",
    "F_avg",
    "F_med",
    "F_var",
    "IC_avg",
    "IC_med",
    "IC_var",
]  # create the variables that you want to store
sonddata_df = pd.DataFrame(
   columns=new_columns
)  # make a dataframe with the same shape as gdf but with above specified columns

En we voeren de eigenlijke analyse uit. Voor elke observatie (veenmonster) wordt de dichtstbijzijnde sondering opgezocht en wordt diens data ingeladen. Na omrekening van zowel de observatie als de sondeerdata naar meter TAW kunnen de Robertson Soil Behavior Type parameters berekend worden. Deze worden toegevoegd aan het resultaat dataframe.

In [ ]:
for observation in set(df_joined['pkey_observatie']):
    nearest_sondering = df_joined[df_joined['pkey_observatie'] == observation].sort_values(by='dist', ascending=False).iloc[0]['pkey_sondering']

    sondering_data = df_sonderingen[
        (df_sonderingen['pkey_sondering'] == nearest_sondering)
        & (df_sonderingen['fs'] > 0)
        & (df_sonderingen['qc'] > 0)
    ].copy()

    observatie_data = df_joined[df_joined['pkey_observatie'] == observation]

    if sondering_data.size == 0:
        continue

    sondering_data["rf"] = sondering_data["fs"] / sondering_data["qc"] / 10  # calculate rf

    # convert depths to relative levels
    sondering_data["mtaw"] = (
        sondering_data["start_sondering_mtaw"] - sondering_data["diepte"]
    )
    sondering_data["mtaw"] = sondering_data["mtaw"].fillna(sondering_data["start_sondering_mtaw"] - sondering_data["lengte"])

    # for the moment there will be no correction with the lag index, this is hard to do automatically, could possibly be done by looking at max curvature or by looping through the indices and finding the first maxima
    sondering_data["unitw"] = 9.81 * (
        0.27 * np.log10(sondering_data["rf"]) + 0.36 * np.log10((sondering_data["qc"] / 0.1)) + 1.236
    )  # calculate unitw

    # if there is no groundwaterdata then take GW equal to start of the sounding
    sondering_data["diepte_gw_m"] = sondering_data["diepte_gw_m"].fillna(sondering_data["start_sondering_mtaw"])

    # if above the groundwater, give a value 0 for pwp
    sondering_data.loc[sondering_data['lengte'] < sondering_data['diepte_gw_m'], 'pwp'] = 0
    # if below GW, give value equal to hydrostatic pwp
    sondering_data.loc[sondering_data['lengte'] >= sondering_data['diepte_gw_m'], 'pwp'] = (sondering_data["lengte"] - sondering_data["diepte_gw_m"]) * 9.81

    sondering_data["stot"] = 0  # make a stot column
    sondering_data['stot'] = (sondering_data['lengte'] - sondering_data['lengte'].shift(1).fillna(0)) * sondering_data['unitw']
    sondering_data['stot'] = sondering_data['stot'].cumsum()

    sondering_data["seff"] = sondering_data["stot"] - sondering_data["pwp"]  # calculate effective stress
    if sondering_data["seff"].isna().any():  # check i used for detecting nan values
        print(f"NaN found for 'seff' in rows {sondering_data[sondering_data['seff'].isna()].index}")
        break

    sondering_data["Qtn"] = ((sondering_data["qc"] * 1000 - sondering_data["stot"]) / 100) * (
        100 / sondering_data["seff"]
    )  # calculate corrected Qtn
    sondering_data["F"] = sondering_data["fs"] * 100 / (sondering_data["qc"] * 1000 - sondering_data["stot"])
    # calculate corrected F-factor
    sondering_data["IC"] = (
        (3.47 - np.log10(sondering_data["Qtn"])) ** 2 + (np.log10(sondering_data["F"]) + 1.22) ** 2
    ) ** 0.5  # Ic for soil behaviour type

    sondering_data["peil"] = (
        sondering_data["start_sondering_mtaw"] - sondering_data["lengte"]
    )  # absolute level

    bovenpeil = (observatie_data['start_boring_mtaw'] - observatie_data["diepte_van_m"]).iloc[0]  # extract boundaries of sample
    onderpeil = (observatie_data['start_boring_mtaw'] - observatie_data["diepte_tot_m"]).iloc[0]  # extract boundaries of sample

    sondering_data_subset = sondering_data[
        (sondering_data["peil"] <= bovenpeil) & (sondering_data["peil"] >= onderpeil)
    ]  # define a subset with the CPT values at the depth of the sample
    if sondering_data_subset.size == 0:  # if this subset is empty just restart loop
        continue
    mean_values = sondering_data_subset.mean(numeric_only=True)  # find mean
    median_values = sondering_data_subset.median(numeric_only=True)  # find median

    if len(sondering_data_subset) > 1:
        variance_values = (
            sondering_data_subset.var(numeric_only=True)
        )  # find variance if there is more than one datapoint
    else:
        variance_values = pd.Series(
            [0 for _ in sondering_data_subset.columns], index=sondering_data_subset.columns
        )  # if this in not the case put a 0 for variance

    new_data = pd.DataFrame(index=observatie_data.index, data={
        'pkey_observatie': observatie_data['pkey_observatie'].iloc[0],
        "qc_avg": mean_values["qc"],
        "fs_avg": mean_values["fs"],
        "rf_avg": mean_values["rf"],
        "Qtn_avg": mean_values["Qtn"],
        "F_avg": mean_values["F"],
        "IC_avg": mean_values["IC"],
        "qc_med": median_values["qc"],
        "fs_med": median_values["fs"],
        "rf_med": median_values["rf"],
        "Qtn_med": median_values["Qtn"],
        "F_med": median_values["F"] ,
        "qc_var": variance_values["qc"],
        "fs_var": variance_values["fs"],
        "rf_var": variance_values["rf"],
        "Qtn_var": variance_values["Qtn"],
        "F_var": variance_values["F"],
        "IC_var": variance_values["IC"],
    })

    sonddata_df = pd.concat([sonddata_df, new_data], ignore_index=True)

sonddata_df


De resultaten worden vervolgens samengevoegd met de brongegevens van de observaties, op basis van de `pkey_observatie`:

In [ ]:
merged_df = pd.merge(geo_df_observaties, sonddata_df, on='pkey_observatie')  # join the two dataframes together
merged_df = merged_df.dropna(
    subset=["Qtn_avg"]
)  # drop all rows with nan values
merged_df

Tenslotte kunnen we deze resultaten weergeven op grafiek:

In [ ]:
# define boundaries for Roberson's graph and plot them
X1 = np.array(
    [
        -3,
        -1,
        -0.890243902439026,
        -0.774390243902441,
        -0.66158536585366,
        -0.548780487804878,
        -0.439024390243903,
        -0.326219512195123,
        -0.210365853658537,
        -0.112804878048781,
        -0.00609756097560987,
        0.082317073170729,
        0.161585365853658,
        0.204268292682926,
        0.225609756097559,
        0.397940008672038,
        0.52244423350632,
        3,
    ]
)
Y1 = np.array(
    [
        1,
        1,
        0.998624484181568,
        0.98624484181568,
        0.961485557083906,
        0.92847317744154,
        0.878954607977991,
        0.812929848693259,
        0.726272352132049,
        0.631361760660247,
        0.50343878954608,
        0.359009628610727,
        0.189821182943603,
        0.0784044016506188,
        0,
        -0.476892499139314,
        -1,
        -1,
    ]
)
X2 = np.array(
    [
        -3,
        0.079181246,
        0.131097560975609,
        0.182926829268291,
        0.246951219512193,
        0.292682926829266,
        0.335365853658536,
        0.38109756097561,
        0.445121951219511,
        0.493902439024389,
        0.551829268292683,
        0.631097560975609,
        0.740853658536585,
        0.838414634146341,
        0.984756097560975,
        1.04878048780488,
        1.17609125905568,
        3,
    ]
)
Y2 = np.array(
    [
        4,
        4,
        3.03301237964236,
        2.88445667125172,
        2.72352132049518,
        2.6079779917469,
        2.50068775790921,
        2.40165061898212,
        2.26134800550206,
        2.15405777166437,
        2.04264099037139,
        1.91884456671252,
        1.82806052269601,
        1.78266850068776,
        1.75378266850069,
        1.7455295735901,
        1.74036268949424,
        1.73639650227664,
    ]
)
X3 = np.array(
    [
        -3,
        -2,
        -1.04575749056068,
        -0.522878745280338,
        -0.0457574905606751,
        0.216463414634144,
        0.420731707317072,
        0.594512195121951,
        0.771341463414634,
        0.887195121951219,
        0.963414634146341,
        0.996951219512195,
        1.04139268515823,
        1.11394335230684,
        1.20411998265592,
        1.252574989,
        1.30102999566398,
        1.47712125471966,
        1.61172330800734,
        3,
    ]
)
Y3 = np.array(
    [
        -0.107504662868674,
        -0.107504662868674,
        -0.099484807248056,
        -0.0784326862439331,
        -0.0182837690892966,
        0.045392022008251,
        0.15268225584594,
        0.292984869325996,
        0.491059147180191,
        0.664374140302613,
        0.808803301237964,
        0.878954607977991,
        0.994223003013752,
        1.19471939352921,
        1.49546397930239,
        1.6959604,
        1.8964567603333,
        2.89893871291058,
        4,
        4,
    ]
)

X4 = np.array(
    [
        -3,
        -2,
        -1,
        -0.522878745280338,
        -0.301029995663981,
        -0.140243902439025,
        -0.0213414634146342,
        0.128048780487803,
        0.277439024390242,
        0.378048780487804,
        0.47560975609756,
        0.570121951219511,
        0.655487804878049,
        0.71951219512195,
        0.774390243902439,
        0.845098040014257,
        0.954242509439325,
        1.07918124604762,
        1.21218760440396,
        3,
    ]
)
Y4 = np.array(
    [
        0.548714105446867,
        0.548714105446867,
        0.567812943589182,
        0.610254806127659,
        0.652696668666137,
        0.664374140302613,
        0.734525447042641,
        0.837689133425034,
        0.969738651994498,
        1.08115543328748,
        1.20082530949106,
        1.34112792297111,
        1.49793672627235,
        1.63823933975241,
        1.80742778541953,
        2.03205720116665,
        2.45647582655142,
        3.09310376462857,
        4,
        4,
    ]
)

X5 = np.array(
    [
        -3,
        -2,
        -1.39794000867204,
        -0.823908740944319,
        -0.503048780487805,
        -0.332317073170732,
        -0.128048780487805,
        0.0579268292682913,
        0.222560975609754,
        0.390243902439023,
        0.506097560975609,
        0.582317073170732,
        0.643292682926829,
        0.658536585365853,
        0.673780487804877,
        0.679878048780487,
        0.700578048780487,
        3,
    ]
)
Y5 = np.array(
    [
        0.793022628459515,
        0.793022628459515,
        0.803892063893693,
        0.84374666048568,
        0.907840440165061,
        0.994497936726272,
        1.11416781292985,
        1.25034387895461,
        1.40302613480055,
        1.60522696011004,
        1.79917469050894,
        1.98899587345254,
        2.30674002751032,
        2.45116918844567,
        2.69050894085282,
        2.98762035763411,
        4,
        4,
    ]
)

X6 = np.array(
    [
        -3,
        -2,
        -1.39794000867204,
        -1,
        -0.771341463414636,
        -0.533536585365855,
        -0.283536585365854,
        -0.0487804878048781,
        0.164634146341461,
        0.314024390243902,
        0.393292682926828,
        0.423780487804877,
        0.544068044350276,
        0.698970004336019,
        0.807589142389764,
        3,
    ]
)
Y6 = np.array(
    [
        1.39511433919444,
        1.39511433919444,
        1.4073041167125,
        1.41953232462173,
        1.4484181568088,
        1.50206327372765,
        1.62173314993122,
        1.76342799356294,
        1.97661623108666,
        2.20357634112792,
        2.36863823933975,
        2.45942228335626,
        2.81319179046214,
        3.42268066636517,
        4,
        4,
    ]
)

X7 = np.array(
    [
        -3,
        -2,
        -1.52287874528034,
        -1.15490195998574,
        -1,
        -0.786585365853661,
        -0.588414634146342,
        -0.39329268292683,
        -0.219512195121951,
        -0.076219512195122,
        0.00609756097560594,
        0.0304878048780472,
        0.113943352306837,
        0.204119982655925,
        0.34409740359441,
        3,
    ]
)
Y7 = np.array(
    [
        2.10124566205107,
        2.10124566205107,
        2.11851911705649,
        2.15306602706735,
        2.17881705639615,
        2.22833562585969,
        2.3191196698762,
        2.45116918844567,
        2.6162310866575,
        2.8101788170564,
        2.97111416781293,
        3.03301237964236,
        3.21538350990112,
        3.47448533498253,
        4,
        4,
    ]
)

Y8 = np.array(
    [
        1,
        1.236584507,
        1.467593452,
        1.641390069,
        2.00087167,
        2.561283296,
        3.04027751,
        3.864751926,
        4.513435735,
        5.329418952,
        5.737406993,
        6.934489818,
        7.871627474,
        9.405725877,
        10.71461614,
        13.26504071,
        14.33512247,
        16.67039577,
        17.48101531,
        20.73321573,
        25.76980375,
        30.8239924,
    ]
)
X8 = np.array(
    [
        5.218732546,
        5.224237832,
        5.235265833,
        5.251851499,
        5.290756014,
        5.324332041,
        5.358121147,
        5.420625748,
        5.45502595,
        5.559541963,
        5.61254852,
        5.885259102,
        6.145248773,
        6.512143221,
        6.934981111,
        7.811089495,
        8.281805716,
        9.300236388,
        9.731586954,
        11.63135731,
        15.24313395,
        19.81818615,
    ]
)

Y9 = np.array(
    [
        1,
        1.24570471,
        1.457672448,
        1.734757,
        2.022753999,
        2.244335431,
        2.546558574,
        2.887030331,
        3.29529913,
        4.377751156,
        5.390771626,
        5.982484067,
        7.152449831,
        7.972379175,
        9.518039505,
        10.38663667,
        11.63636757,
        12.44098606,
        13.1809551,
        14.21891386,
        15.08325454,
        16.11449903,
        16.95549455,
        19.15734093,
        20.67643722,
        22.67543126,
        25.76980375,
        31.22082999,
    ]
)
X9 = np.array(
    [
        2.359210475,
        2.381703987,
        2.412029295,
        2.450479487,
        2.481680486,
        2.521240988,
        2.572257547,
        2.660524333,
        2.751819986,
        3.00507027,
        3.314662302,
        3.499621817,
        3.91963219,
        4.195462425,
        4.867922672,
        5.240788561,
        5.866673096,
        6.2441507,
        6.626424983,
        7.192139126,
        7.700695838,
        8.334363447,
        8.93498453,
        10.23676764,
        11.22018726,
        12.69483156,
        15.00257488,
        19.66118419,
    ]
)

Y10 = np.array(
    [
        1,
        1.490163293,
        1.722664767,
        1.902319213,
        2.331621252,
        2.669482766,
        2.950129966,
        3.29529913,
        3.574741878,
        3.924187211,
        4.578962969,
        4.912810557,
        5.593309253,
        6.119288823,
        6.644642623,
        7.023246768,
        7.361639679,
        8.081269437,
        8.575422275,
        10.30942724,
        11.15336857,
        12.18152662,
        12.75222616,
        13.7914596,
        14.48171973,
        15.39332793,
        16.55068513,
        17.67101739,
        18.40892945,
        20.02936469,
        21.64511984,
        22.72845428,
        23.96333726,
        25.50421875,
        27.14418142,
        28.48824655,
        30.28155009,
    ]
)
X10 = np.array(
    [
        0.695855433,
        0.743646766,
        0.788045272,
        0.821991165,
        0.916285475,
        0.991683911,
        1.058117967,
        1.143377667,
        1.221904814,
        1.336468837,
        1.518324783,
        1.624314987,
        1.896625055,
        2.063542636,
        2.279566394,
        2.419670763,
        2.561432124,
        2.846248441,
        3.057806323,
        3.908489836,
        4.312064256,
        4.914335096,
        5.169445303,
        5.837057497,
        6.223493829,
        6.785502665,
        7.532065355,
        8.288357264,
        8.822649179,
        9.933738163,
        11.18475323,
        12.03638898,
        13.01858971,
        14.26925174,
        15.81417007,
        16.90919862,
        18.5825382,
    ]
)

plt.plot(
    10**X1, 10**Y1, color="grey", linestyle="-", linewidth=1, label="Robertson's zones"
)
plt.plot(10**X2, 10**Y2, color="grey", linestyle="-", linewidth=1)
plt.plot(10 ** X3[5:16], 10 ** Y3[5:16], color="grey", linestyle="-", linewidth=1)
plt.plot(10 ** X4[5:15], 10 ** Y4[5:15], color="grey", linestyle="-", linewidth=1)
plt.plot(10 ** X5[4:17], 10 ** Y5[4:17], color="grey", linestyle="-", linewidth=1)
plt.plot(10 ** X6[0:11], 10 ** Y6[0:11], color="grey", linestyle="-", linewidth=1)
plt.plot(10 ** X7[0:13], 10 ** Y7[0:13], color="grey", linestyle="-", linewidth=1)
plt.plot(
    X8,
    Y8,
    color="red",
    linestyle="-",
    linewidth=1,
    label="Possibly peat / Possibly clay",
)
# plt.plot(X9,Y9, color='black',linestyle='-', linewidth=1, label="Lengkeek & Brinkgreve zone 2B")
plt.plot(
    X10,
    Y10,
    color="purple",
    linestyle="-",
    linewidth=1,
    label="Possibly slightly peaty clay / Possibly clay",
)


# Customize the plot
plt.title("Robertson's Soil Identification Chart")
plt.xlabel("F")
plt.ylabel("Qtn")
plt.grid(True)
plt.xscale("log")
plt.yscale("log")

# Set custom x and y axis labels
x_labels = [0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000]
y_labels = [0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000]
plt.xticks(x_labels)
plt.yticks(y_labels)
plt.xlim([0.1, 15])
plt.ylim([1, 1000])

# Average values from your DataFrame (replace with your actual values)
avg_Qtn = merged_df["Qtn_avg"]
avg_F = merged_df["F_avg"]

# just plot the dots
plt.scatter(avg_F, avg_Qtn, label="Pydov_data")
plt.legend()